# Use Case 3 — Self-Healing & Closed Loop (real DPO retrain -> gate -> deploy)

The DECIDE closed loop for real: retrain_job really trains via build_retrainer(...) -> a real TRL DPOTrainer (LoRA on SmolLM2-360M) -> a real adapter on disk; the deployment gate really decides on real task accuracy (old vs retrained model); apply_decision really deploys via deploy_trained_model, rewriting config.yaml; then the deployed adapter is read back and reloaded as a real PeftModel -- literal proof a real adapter was produced and deployed. This replaces an earlier version of this notebook that used a simplified stand-in trainer instead of the real closed-loop retrain path.

In [ ]:
"""The DECIDE closed loop for real — retrain → gate → deploy a real adapter.

`FullClosedLoop` is the flagship DECIDE closed-loop orchestrator: production failures →
detect/classify/generate → buffer → trigger → **background retrain → deployment gate → deploy**.
Every model-dependent boundary is an injected callable so the loop is unit-testable GPU-free —
and in every test/notebook those boundaries are **stubs**: `_stub_retrain_job` returns
`{"model_path": "/tmp/stub_model"}`, verdict runners are fakes. So the load-bearing chain the loop
exists for — *retrain → gate → deploy a real adapter* — has never actually run end-to-end.

This runs that chain for real:

  * `retrain_job` really trains: `build_retrainer(...)` → real TRL `DPOTrainer` (LoRA on
    `SmolLM2-360M`) → `save_model` → a real adapter on disk (asserted present);
  * the **deployment gate really decides** on real task accuracy — the old (base) model vs the new
    (retrained) model are scored by real verdict runners over a non-empty test set the gate
    reconstructs from a real Decide audit log;
  * `apply_decision` really deploys via the real `deploy_trained_model` bridge, which rewrites a
    (scratch) `config.yaml`;
  * we then **read `config.yaml` back and load the deployed adapter** as a real `PeftModel` — the
    literal proof that a real adapter was produced *and* deployed.

Scope: this focuses on the never-real **retrain → gate → deploy** chain. Live-LLM
classify/generate is already proven for real in another notebook in this set, so here we submit
real `TrainingExample`s straight into the loop's real buffer (`runner.submit`) rather than
re-standing-up a local LLM — that boundary is explicit, not hidden. The gate runs on **task
accuracy** (its primary signal); the optional trajectory signal (which would call litellm) is left
off. A fully-live-detection variant is a straightforward extension on request.

The gate's "verdict" here is **contract-format adherence**: does the model emit the `SENTIMENT=`
form the DPO step teaches (chosen uses it, rejected is prose)? The base model mostly answers in
prose; the retrained one emits the form — so the gate sees a real accuracy gain (≈0.33 → 1.00) and
approves on merit, not a default-approve on an empty test set. Getting the exact *label* right is a
separate, harder axis and is deliberately not claimed (six preference pairs on a 360M model teach
the format, not always the correct label — an honest limit).

Requires a CUDA GPU and `HuggingFaceTB/SmolLM2-360M-Instruct` in the local HF cache. No network.
"""
from __future__ import annotations

In [2]:
import os

In [3]:
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

'false'

In [4]:
import glob
import json
import re
import tempfile

In [5]:
import yaml

In [6]:
from agenttune.decide.closed_loop.contracts import TrainingExample
from agenttune.decide.closed_loop.full_loop import FullClosedLoop, PathAConfig, GateConfig
from agenttune.decide.closed_loop.retrain_config import RetrainConfig, build_retrainer
from agenttune.decide.closed_loop.retraining_trigger import RetrainingTrigger, TriggerConfig

In [7]:
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"

In [8]:
# The contract the DPO step teaches: answer exactly `SENTIMENT=<label>`.
REVIEWS = [
    ("the delivery was late and support ignored me", "negative"),
    ("fantastic product, works perfectly out of the box", "positive"),
    ("terrible experience, it broke on day one", "negative"),
    ("absolutely love it, would recommend to anyone", "positive"),
    ("the refund never arrived and nobody replied", "negative"),
    ("smooth setup and great battery life", "positive"),
]
# The gate's "verdict" is contract-*format* adherence: did the model emit the `SENTIMENT=` form
# the DPO step teaches (chosen uses it, rejected is prose)? That is what a short DPO run on six
# pairs reliably changes — the base model answers in prose, the retrained one emits the form.
# Getting the exact *label* right is a separate, harder axis (imitation/SFT territory) and is NOT
# claimed here; measuring format keeps the gate honest and driven by a real behavioural shift.
CONTRACT_RE = re.compile(r"SENTIMENT\s*=", re.IGNORECASE)

In [9]:
def _prompt(review: str) -> str:
    return f"Classify sentiment. Reply with exactly SENTIMENT=<label>. Review: {review}"

--------------------------------------------------------------------------
real Decide audit log -> the gate's ground-truth test set
--------------------------------------------------------------------------

In [10]:
def write_audit_log(path: str) -> int:
    """Write a real Decide audit.jsonl of successful runs. Each pipeline gets a per-stage
    entry (carrying the review prompt as input) and a passing completion entry — exactly the
    two shapes `DeploymentGate.build_test_set` reconstructs test cases from."""
    lines = []
    for i, (review, _label) in enumerate(REVIEWS):
        pid = f"pipe-{i}"
        lines.append({"pipeline_id": pid, "stage_id": "classify", "stage_type": "llm",
                      "input": _prompt(review), "output": "SENTIMENT=...", "reward": 1.0})
        lines.append({"pipeline_id": pid, "template_id": "sentiment", "verdict": "PASS",
                      "is_complete": True, "error": None, "step_count": 1, "episode_reward": 1.0})
    with open(path, "w", encoding="utf-8") as f:
        for rec in lines:
            f.write(json.dumps(rec) + "\n")
    return len(REVIEWS)

--------------------------------------------------------------------------
real preference examples (what real live-LLM detection would have produced)
--------------------------------------------------------------------------

In [11]:
def make_examples() -> list[TrainingExample]:
    out = []
    for i, (review, label) in enumerate(REVIEWS):
        out.append(TrainingExample(
            trajectory_id=f"pipe-{i}",
            original_failure_type="format_violation",
            root_cause="answered in prose instead of the SENTIMENT= contract",
            prompt=[{"role": "user", "content": _prompt(review)}],
            chosen=[{"role": "assistant", "content": f"SENTIMENT={label}"}],
            rejected=[{"role": "assistant", "content": f"The sentiment of the review is {label}."}],
            salvaged_at_attempt=1,
        ))
    return out

--------------------------------------------------------------------------
real retrain job: build_retrainer -> train -> save -> return {"path": ...}
--------------------------------------------------------------------------

In [12]:
def make_retrain_job(output_dir: str):
    def retrain_job(examples):
        cfg = RetrainConfig(
            model=MODEL, algorithm="dpo", output_dir=output_dir,
            lora_r=16, lora_alpha=32, num_train_epochs=30,
            per_device_train_batch_size=3, learning_rate=5e-4, beta=0.1,
            max_steps=60, seed=0, extra={"report_to": "none"},
        )
        trainer = build_retrainer(examples, cfg)
        stats = trainer.train()
        trainer.save_model(output_dir)               # persist the adapter
        # the loop reads result["path"]; assert the adapter is really on disk first
        assert os.path.exists(os.path.join(output_dir, "adapter_config.json")), "adapter not saved"
        assert os.path.exists(os.path.join(output_dir, "adapter_model.safetensors")), "weights not saved"
        return {"path": output_dir,
                "final_loss": stats.get("final_loss"),
                "total_steps": stats.get("total_steps")}
    return retrain_job

--------------------------------------------------------------------------
real verdict runners: input_text -> "PASS"/"FAIL" (contract adherence)
--------------------------------------------------------------------------

In [13]:
def make_verdict_runner(adapter_path: str | None):
    """Load SmolLM2-360M (optionally + a LoRA adapter) once and return an
    input_text -> verdict function. Verdict is 'PASS' iff the greedy generation follows the
    SENTIMENT=<label> contract, else 'FAIL'. Greedy → reproducible."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
    if adapter_path is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path).to("cuda")
    model.eval()

    def verdict(input_text):
        messages = [{"role": "user", "content": str(input_text)}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = model.generate(**ids, max_new_tokens=16, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        text = tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        return "PASS" if CONTRACT_RE.search(text) else "FAIL"

    return verdict

In [ ]:
def main() -> None:
    scratch = tempfile.mkdtemp(prefix="closed_loop_real_")
    audit_path = os.path.join(scratch, "audit.jsonl")
    config_path = os.path.join(scratch, "config.yaml")
    adapter_dir = os.path.join(scratch, "retrained_adapter")

    n = write_audit_log(audit_path)
    print(f"[audit]   wrote {n} successful pipelines -> {os.path.basename(audit_path)} "
          f"(the gate's ground-truth test set)")

    # seed a minimal Decide config for the deploy bridge to rewrite
    with open(config_path, "w") as f:
        yaml.safe_dump({"default_model": MODEL, "backend": "transformers", "stages": {}}, f)

    # old (currently-deployed) model = base; the gate's A/B baseline
    print("[baseline] loading base verdict runner (the currently-deployed model)...")
    old_runner = make_verdict_runner(adapter_path=None)

    loop = FullClosedLoop(
        path_a=PathAConfig(audit_log_path=audit_path,
                           classifier_model="unused-no-live-path-a",
                           generator_model="unused-no-live-path-a"),
        retrain_job=make_retrain_job(adapter_dir),
        build_model_runner=lambda path: make_verdict_runner(adapter_path=path),
        old_model_runner=old_runner,
        # low thresholds so a handful of real examples fires T1 deterministically
        trigger=RetrainingTrigger(TriggerConfig(total_failures_threshold=len(REVIEWS),
                                                min_examples_ready=len(REVIEWS))),
        gate_cfg=GateConfig(config_path=config_path, backend="transformers",
                            task_regression_tol=0.0, min_test_samples=3),
        collect_trajectories=None,               # gate on task accuracy (primary signal)
    )
    print(f"[gate]    built test set: {len(loop.test_set)} cases from the audit log")

    # submit real preference examples into the real buffer (detection boundary, stated)
    for ex in make_examples():
        loop.runner.submit(ex)
    print(f"[buffer]  submitted {len(REVIEWS)} real TrainingExamples")

    # one control cycle: fires the trigger -> background retrain -> gate -> deploy
    rec = loop.tick()
    print(f"[tick]    trigger fired={rec.fired} reason={rec.reason!r} -> background retrain")
    loop.wait_for_retrain()                      # gate runs on the daemon thread; wait for it

    cycle = loop.cycles[-1]
    d = cycle.decision
    print(f"[retrain] success={cycle.retrain_success}")
    print(f"[gate]    task accuracy  old={d.task_old:.2f}  new={d.task_new:.2f}  "
          f"delta={d.task_delta:+.2f}  over {d.num_cases} cases")
    print(f"[gate]    decision: approved={d.approved}  ({d.reason})")
    print(f"[apply]   {cycle.applied}")

    # literal proof the adapter was produced AND deployed AND is loadable
    with open(config_path) as f:
        deployed = yaml.safe_load(f)
    adapter_files = sorted(os.path.basename(p) for p in glob.glob(adapter_dir + "/*")
                           if os.path.basename(p).startswith("adapter"))
    deployed_ok = deployed.get("default_model") == adapter_dir
    print(f"[deploy]  config.yaml default_model -> {os.path.basename(deployed.get('default_model',''))!r} "
          f"(points at the new adapter: {deployed_ok})")
    print(f"[deploy]  adapter files on disk: {adapter_files}")

    loadable = False
    if deployed_ok:
        import torch
        from transformers import AutoModelForCausalLM
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
        reloaded = PeftModel.from_pretrained(base, deployed["default_model"])
        loadable = type(reloaded).__name__ == "PeftModelForCausalLM"
        print(f"[deploy]  reloaded the deployed adapter as {type(reloaded).__name__}: {loadable}")

    ok = (cycle.retrain_success and d.approved and cycle.applied == "deployed"
          and d.task_new > d.task_old and deployed_ok and loadable)
    print(f"[verdict] retrain→gate→deploy ran end-to-end; gate approved a REAL accuracy gain "
          f"({d.task_old:.2f}→{d.task_new:.2f}) and a real adapter is deployed+loadable: {ok}")

In [15]:
if __name__ == "__main__":
    main()

[audit]   wrote 6 successful pipelines -> audit.jsonl (the gate's ground-truth test set)
[baseline] loading base verdict runner (the currently-deployed model)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[gate]    built test set: 6 cases from the audit log
[buffer]  submitted 6 real TrainingExamples
[tick]    trigger fired=True reason='total_failures_exceeded' -> background retrain


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Step,Training Loss
10,0.527552
20,0.026491
30,0.000715
40,0.000158
50,0.000089
60,0.000077


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✓ Backup created: /tmp/closed_loop_real_8_1uer1c/config.yaml.backup
✓ Model deployed: /tmp/closed_loop_real_8_1uer1c/retrained_adapter
✓ Backend: transformers
✓ Config updated: /tmp/closed_loop_real_8_1uer1c/config.yaml
[retrain] success=True
[gate]    task accuracy  old=0.33  new=1.00  delta=+0.67  over 6 cases
[gate]    decision: approved=True  (approved: no regression on task or trajectory)
[apply]   deployed
[deploy]  config.yaml default_model -> 'retrained_adapter' (points at the new adapter: True)
[deploy]  adapter files on disk: ['adapter_config.json', 'adapter_model.safetensors']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[deploy]  reloaded the deployed adapter as PeftModelForCausalLM: True
[verdict] retrain→gate→deploy ran end-to-end; gate approved a REAL accuracy gain (0.33→1.00) and a real adapter is deployed+loadable: True
